In [18]:
import numpy as np
import pandas as pd
import pickle, re
import json
from pathlib import Path

In [4]:
def load_sid_split(in_json):
    """
    Load train/test subject IDs from a JSON.
    """
    with open(in_json, "r", encoding="utf-8") as f:
        obj = json.load(f)
    train = set(obj["train_sids"])
    test  = set(obj["test_sids"])
    if train & test:
        raise ValueError("Loaded split has overlapping sids.")
    return train, test, obj.get("meta", {})

# --- APPLY TO ANY DATAFRAME ---

def apply_sid_split(data, train_sids, test_sids, sid_col="sid", sex_col="sex"):
    """
    Given a DataFrame and saved subject IDs, return aligned splits for all/male/female.
    """
    sid_as_str = data[sid_col].astype(str)
    is_train = sid_as_str.isin(train_sids)
    is_test  = sid_as_str.isin(test_sids)

    train_all = data[is_train].copy()
    test_all  = data[is_test].copy()

    male   = data[data[sex_col] == "M"]
    female = data[data[sex_col] == "F"]

    train_m = male[male[sid_col].astype(str).isin(train_sids)].copy()
    test_m  = male[male[sid_col].astype(str).isin(test_sids)].copy()
    train_f = female[female[sid_col].astype(str).isin(train_sids)].copy()
    test_f  = female[female[sid_col].astype(str).isin(test_sids)].copy()

    return {"all": (train_all, test_all),
            "male": (train_m, test_m),
            "female": (train_f, test_f)}

In [12]:
import pickle
# --- your existing helper (kept here for completeness) ---
def predict_next_x(model, x_t, dt=None):
    if dt is None:
        X = np.array([[x_t]], float)
    else:
        X = np.array([[x_t, dt]], float)
    return float(model.predict(X)[0])

def load_model(name):
    """
    Load a model saved by save_model(...).
    Returns (model, meta_dict_or_None).
    """
    p = Path(name)
    with open(p, "rb") as f:
        bundle = pickle.load(f)
    return bundle["model"], bundle.get("meta")

# --- build skip-1 triplets: (t, t+1, t+2) ---
def build_skip1_triplets(df_long, sid_col="sid", landmark_col="landmark", age_col="age", x_col="x"):
    d = df_long[[sid_col, landmark_col, age_col, x_col]].dropna().copy()
    d[age_col] = pd.to_numeric(d[age_col], errors="coerce")
    d[x_col]   = pd.to_numeric(d[x_col], errors="coerce")
    d = d.dropna(subset=[age_col, x_col])

    rows = []
    for (sid, lmk), g in d.groupby([sid_col, landmark_col], sort=False):
        g = g.sort_values(age_col)
        ages = g[age_col].to_numpy()
        xs   = g[x_col].to_numpy()
        if len(xs) < 3: 
            continue
        dt = np.diff(ages)
        for i in range(len(xs)-2):
            dt1, dt2 = dt[i], dt[i+1]
            if not np.isfinite([xs[i], xs[i+1], xs[i+2], dt1, dt2]).all(): 
                continue
            if dt1 <= 0 or dt2 <= 0: 
                continue
            rows.append((sid, lmk, ages[i], ages[i+1], ages[i+2], xs[i], xs[i+1], xs[i+2], dt1, dt2))

    cols = [sid_col, landmark_col, f"{age_col}_t", f"{age_col}_t1", f"{age_col}_t2",
            "x_t", "x_t1", "x_t2", "dt1", "dt2"]
    return pd.DataFrame(rows, columns=cols)


# ---- new: infer (landmark, sex, axis) from filename like "u 6 apex_female_y_.pkl" ----
def infer_meta_from_filename(pkl_path: str):
    """
    Parse filenames like:
        "u 6 apex_female_y_.pkl"
        "porion_male_x.pkl"
        "u 6 cusp_both_y_.pkl"
    Returns: (landmark, sex_key, axis)
    """
    base = Path(pkl_path).name  # basename only (no folders)
    # landmark can have spaces/underscores; optional trailing "_" before .pkl
    m = re.match(r"^(.+?)_(male|female|both)_(x|y)_?\.pkl$", base, flags=re.IGNORECASE)
    if not m:
        raise ValueError(
            f"Filename must look like 'landmark_(male|female|both)_(x|y)[_].pkl', got: {base}"
        )
    landmark = m.group(1).strip()
    sex_key  = m.group(2).lower()
    axis     = m.group(3).lower()
    return landmark, sex_key, axis

def subset_for_model(df, landmark, sex_key, landmark_col="landmark", sex_col="sex"):
    df = df.copy()
    # landmark must match exactly (your data likely has names like "u 6 apex")
    df = df[df[landmark_col] == landmark]
    # sex filter
    if sex_key in ("male", "female"):
        want = "M" if sex_key == "male" else "F"
        df = df[df[sex_col].astype(str).str.upper() == want]
    # if 'both', no sex filter
    return df


# ---- core: evaluate skip-1 MAE on the filtered subset ----
def eval_skip1_mae(model, df_test, axis="x", use_dt=True,
                   sid_col="sid", landmark_col="landmark", age_col="age"):
    x_col = axis  # 'x' or 'y'
    trip = build_skip1_triplets(df_test, sid_col=sid_col, landmark_col=landmark_col, age_col=age_col, x_col=x_col)
    if trip.empty:
        raise ValueError("No (t, t+1, t+2) triplets after filtering (sex+landmark).")
    preds = []
    for _, r in trip.iterrows():
        xhat_t1 = predict_next_x(model, float(r["x_t"]), float(r["dt1"]) if use_dt else None)
        xhat_t2 = predict_next_x(model, xhat_t1,            float(r["dt2"]) if use_dt else None)
        preds.append(xhat_t2)
    trip["xhat_t2_skip1"] = preds
    mae = float(np.mean(np.abs(trip["xhat_t2_skip1"] - trip["x_t2"])))
    return mae, len(trip), trip

# ---- convenience: load model, infer meta, filter, eval ----
def eval_skip1_mae_from_file(pkl_path, df_test, use_dt=True,
                             sid_col="sid", landmark_col="landmark", sex_col="sex", age_col="age"):
    
    model, meta = load_model(pkl_path)
    landmark, sex_key, axis = infer_meta_from_filename(pkl_path)
    df_sub = subset_for_model(df_test, landmark, sex_key, landmark_col=landmark_col, sex_col=sex_col)
    mae, n, details = eval_skip1_mae(model, df_sub, axis=axis, use_dt=use_dt,
                                     sid_col=sid_col, landmark_col=landmark_col, age_col=age_col)
    print(f"[{landmark} | {sex_key} | axis={axis}]  Skip-1 MAE = {mae:.4f} over {n} triplets")
    return mae, n, details




In [35]:
data = pd.read_csv("/data/all_landmark_series_long.csv")
train_sids, test_sids, meta = load_sid_split("/data/splits/sid_split_v1.json")
re_splits = apply_sid_split(data, train_sids, test_sids)

train_all, test_all = re_splits["all"]
train_m, test_m     = re_splits["male"]
train_f, test_f     = re_splits["female"]

In [ ]:
landmarks = ['sella','nasion','porion','orbitale','u i apex','point a','u i edge','l i edge','point b','l i apex','pogonion','menton','u 6 apex','u 6 cusp','l 6 cusp','l 6 apex','gonion l','gonion u','condyle','pns','basion','u_6_mcp','l_6_mcp','ans','articular','mid gonion']

In [32]:
import os

In [37]:
import os
import pandas as pd

# containers
male_x_rows, male_y_rows   = [], []
female_x_rows, female_y_rows = [], []
both_x_rows, both_y_rows   = [], []

models_dir = "./models"

for fname in os.listdir(models_dir):
    if not fname.lower().endswith(".pkl"):
        continue
    # parse meta from *basename*
    try:
        landmark, sex, axis = infer_meta_from_filename(fname)  # uses Path(...).name inside
    except ValueError:
        print(f"Skip (pattern mismatch): {fname}")
        continue

    # choose test split and target table by sex+axis
    if sex == "male":
        data = test_m
        target_rows = male_x_rows if axis == "x" else male_y_rows
    elif sex == "female":
        data = test_f
        target_rows = female_x_rows if axis == "x" else female_y_rows
    elif sex == "both":
        data = test_all
        target_rows = both_x_rows if axis == "x" else both_y_rows
    else:
        continue

    pkl_path = os.path.join(models_dir, fname)
    try:
        mae, n, _ = eval_skip1_mae_from_file(pkl_path, data)  # uses axis from filename internally
        target_rows.append({"landmark": landmark, "mae": mae, "n": n})
    except Exception as e:
        print(f"Skipping {fname}: {e}")

# build DFs
male_x   = pd.DataFrame(male_x_rows,   columns=["landmark", "mae", "n"]).sort_values("landmark")
male_y   = pd.DataFrame(male_y_rows,   columns=["landmark", "mae", "n"]).sort_values("landmark")
female_x = pd.DataFrame(female_x_rows, columns=["landmark", "mae", "n"]).sort_values("landmark")
female_y = pd.DataFrame(female_y_rows, columns=["landmark", "mae", "n"]).sort_values("landmark")
both_x   = pd.DataFrame(both_x_rows,   columns=["landmark", "mae", "n"]).sort_values("landmark")
both_y   = pd.DataFrame(both_y_rows,   columns=["landmark", "mae", "n"]).sort_values("landmark")

# save
male_x.to_csv("skip1_male_x_results.csv", index=False)
male_y.to_csv("skip1_male_y_results.csv", index=False)
female_x.to_csv("skip1_female_x_results.csv", index=False)
female_y.to_csv("skip1_female_y_results.csv", index=False)
both_x.to_csv("skip1_both_x_results.csv", index=False)
both_y.to_csv("skip1_both_y_results.csv", index=False)


[ans | both | axis=x]  Skip-1 MAE = 1.5751 over 268 triplets
[ans | both | axis=y]  Skip-1 MAE = 1.5050 over 268 triplets
[ans | female | axis=x]  Skip-1 MAE = 1.6162 over 136 triplets
[ans | female | axis=y]  Skip-1 MAE = 1.5767 over 136 triplets
[ans | male | axis=x]  Skip-1 MAE = 1.5327 over 132 triplets
[ans | male | axis=y]  Skip-1 MAE = 1.4311 over 132 triplets
[articular | both | axis=x]  Skip-1 MAE = 1.3117 over 266 triplets
[articular | both | axis=y]  Skip-1 MAE = 1.5146 over 269 triplets
[articular | female | axis=x]  Skip-1 MAE = 1.1845 over 135 triplets
[articular | female | axis=y]  Skip-1 MAE = 1.5325 over 136 triplets
[articular | male | axis=x]  Skip-1 MAE = 1.4428 over 131 triplets
[articular | male | axis=y]  Skip-1 MAE = 1.4962 over 133 triplets
[basion | both | axis=x]  Skip-1 MAE = 1.5895 over 268 triplets
[basion | both | axis=y]  Skip-1 MAE = 1.7602 over 268 triplets
[basion | female | axis=x]  Skip-1 MAE = 1.4528 over 136 triplets
[basion | female | axis=y]  Sk

In [31]:
mae, n, details = eval_skip1_mae_from_file("./models/porion_female_y_.pkl", test_f)
print(f"Skip-1 MAE = {mae:.4f} over {n} triplets")
details.head()

[porion | female | axis=y]  Skip-1 MAE = 2.5469 over 134 triplets
Skip-1 MAE = 2.5469 over 134 triplets


,sid,landmark,age_t,age_t1,age_t2,x_t,x_t1,x_t2,dt1,dt2,xhat_t2_skip1
0,007,porion,8.167,9.167,10.167,-18.2,-15.8,-20.3,1.000,1.000,-19.045768
1,007,porion,9.167,10.167,11.250,-15.8,-20.3,-20.2,1.000,1.083,-17.228235
2,007,porion,10.167,11.250,12.250,-20.3,-20.2,-20.7,1.083,1.000,-20.643689
3,007,porion,11.250,12.250,13.583,-20.2,-20.7,-22.5,1.000,1.333,-20.581410
4,008,porion,6.667,7.667,8.667,-16.2,-10.8,-14.5,1.000,1.000,-17.527540


In [47]:
import pandas as pd

# 1) Load base ordering (female_x.csv) and all result CSVs
base_landmarks_serial = pd.read_csv("female_x.csv")  # reference order
male_x   = pd.read_csv("skip1_male_x_results.csv")
male_y   = pd.read_csv("skip1_male_y_results.csv")
female_x = pd.read_csv("skip1_female_x_results.csv")
female_y = pd.read_csv("skip1_female_y_results.csv")
both_x   = pd.read_csv("skip1_both_x_results.csv")
both_y   = pd.read_csv("skip1_both_y_results.csv")

def normalize_landmark(df):
    if "landmark" in df.columns:
        df["landmark"] = df["landmark"].astype(str).str.strip()
    return df

base_landmarks_serial = normalize_landmark(base_landmarks_serial)
male_x   = normalize_landmark(male_x)
male_y   = normalize_landmark(male_y)
female_x = normalize_landmark(female_x)
female_y = normalize_landmark(female_y)
both_x   = normalize_landmark(both_x)
both_y   = normalize_landmark(both_y)

def reorder_like(base_df: pd.DataFrame, df: pd.DataFrame, key="landmark") -> pd.DataFrame:
    base_order = base_df[key].tolist()
    aligned = pd.DataFrame({key: base_order}).merge(df, on=key, how="left")
    extras = df[~df[key].isin(base_order)]
    out = pd.concat([aligned, extras], ignore_index=True)
    return out

# 2) Reorder using female_x as the serial
male_x_serial   = reorder_like(base_landmarks_serial, male_x)
male_y_serial   = reorder_like(base_landmarks_serial, male_y)
female_x_serial = reorder_like(base_landmarks_serial, female_x)
female_y_serial = reorder_like(base_landmarks_serial, female_y)
both_x_serial   = reorder_like(base_landmarks_serial, both_x)
both_y_serial   = reorder_like(base_landmarks_serial, both_y)

# 3) Optional: round MAE to N digits (set to None to skip)
ROUND_MAE = 4  # <- change to None to disable rounding
tables = [male_x_serial, male_y_serial, female_x_serial, female_y_serial, both_x_serial, both_y_serial]
if ROUND_MAE is not None:
    for _df in tables:
        if "mae" in _df.columns:
            _df["mae"] = _df["mae"].round(ROUND_MAE)

# 4) Save to one Excel workbook
with pd.ExcelWriter("skip1_results_serialized_by_female_x.xlsx") as writer:
    male_x_serial.to_excel(writer,   sheet_name="male_x",   index=False)
    male_y_serial.to_excel(writer,   sheet_name="male_y",   index=False)
    female_x_serial.to_excel(writer, sheet_name="female_x", index=False)
    female_y_serial.to_excel(writer, sheet_name="female_y", index=False)
    both_x_serial.to_excel(writer,   sheet_name="both_x",   index=False)
    both_y_serial.to_excel(writer,   sheet_name="both_y",   index=False)

print("Saved: skip1_results_serialized_by_female_x.xlsx (MAE rounded to", ROUND_MAE, "digits)")


Saved: skip1_results_serialized_by_female_x.xlsx (MAE rounded to 4 digits)


## Skip more

In [10]:
import json
def load_sid_split(in_json):
    """
    Load train/test subject IDs from a JSON.
    """
    with open(in_json, "r", encoding="utf-8") as f:
        obj = json.load(f)
    train = set(obj["train_sids"])
    test  = set(obj["test_sids"])
    if train & test:
        raise ValueError("Loaded split has overlapping sids.")
    return train, test, obj.get("meta", {})

# --- APPLY TO ANY DATAFRAME ---

def apply_sid_split(data, train_sids, test_sids, sid_col="sid", sex_col="sex"):
    """
    Given a DataFrame and saved subject IDs, return aligned splits for all/male/female.
    """
    sid_as_str = data[sid_col].astype(str)
    is_train = sid_as_str.isin(train_sids)
    is_test  = sid_as_str.isin(test_sids)

    train_all = data[is_train].copy()
    test_all  = data[is_test].copy()

    male   = data[data[sex_col] == "M"]
    female = data[data[sex_col] == "F"]

    train_m = male[male[sid_col].astype(str).isin(train_sids)].copy()
    test_m  = male[male[sid_col].astype(str).isin(test_sids)].copy()
    train_f = female[female[sid_col].astype(str).isin(train_sids)].copy()
    test_f  = female[female[sid_col].astype(str).isin(test_sids)].copy()

    return {"all": (train_all, test_all),
            "male": (train_m, test_m),
            "female": (train_f, test_f)}

In [5]:
import re
from pathlib import Path
from glob import glob
import numpy as np
import pandas as pd

def _predict_nextstep(model, pairs: pd.DataFrame, use_dt: bool = True) -> pd.DataFrame:
    """
    One-step prediction using your trained sklearn model/pipeline.
    Expects 'pairs' with columns: ['sid','age_next','val_t',('dt'),('y_true')].
    Returns columns: ['sid','age_next','y_true','y_pred'].
    """
    if pairs is None or pairs.empty:
        return pd.DataFrame(columns=["sid", "age_next", "y_true", "y_pred"])

    pairs = pairs.copy()
    feat_cols = ["val_t"]
    if use_dt:
        if "dt" not in pairs.columns:
            # if dt isn't provided, default to zeros (keeps shape consistent)
            pairs["dt"] = 0.0
        feat_cols.append("dt")

    X = pairs[feat_cols].astype(float).to_numpy()
    y_pred = model.predict(X)

    out = pairs[["sid", "age_next"]].copy()
    out["y_true"] = pairs["y_true"] if "y_true" in pairs.columns else np.nan
    out["y_pred"] = np.asarray(y_pred, dtype=float)
    return out


def _find_twin_model(models_dir: str, landmark: str, sex_key: str, other_axis: str) -> str :
    """
    Find the counterpart model file (x<->y) for the SAME (landmark, sex).
    Searches *.pkl in models_dir using your infer_meta_from_filename().
    Returns the file path or None if not found.
    """
    def _norm(s: str) -> str:
        # normalize spaces/underscores/case for robust matching
        return re.sub(r"[ _]+", " ", str(s).strip().lower())

    want_lm  = _norm(landmark)
    want_sex = str(sex_key).lower()
    want_ax  = other_axis.lower()

    best = None
    for fp in sorted(glob(str(Path(models_dir) / "*.pkl"))):
        try:
            lm, sex, ax = infer_meta_from_filename(Path(fp).name)
        except Exception:
            continue
        if ax.lower() != want_ax:
            continue
        if _norm(lm) == want_lm and str(sex).lower() == want_sex:
            # pick the shortest filename if multiple (usually the cleanest match)
            if best is None or len(Path(fp).name) < len(Path(best).name):
                best = fp

    return best


In [6]:
import numpy as np
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------------------
# Core: vectorized k-step simulation with your existing one-step predictor
# ---------------------------------------------------------------------
def _kstep_predict_axis(model, df_sub, axis_col="x", k=1, use_dt=True):
    """
    Use the same one-step model repeatedly to predict k steps ahead.
    Returns: DataFrame with columns [sid, age_target, y_true, y_pred]
    """
    out = []
    by_sid = df_sub.groupby("sid", sort=False)
    for sid, g in by_sid:
        g = g[["age", axis_col]].dropna().sort_values("age")
        if len(g) < (k + 1):
            continue

        ages = g["age"].to_numpy(float)
        vals = g[axis_col].to_numpy(float)
        dts  = np.diff(ages)

        # start positions that have k-step targets
        starts = np.arange(0, len(vals) - k, dtype=int)

        # current predicted values (vector) initialized with observed at t
        cur = vals[starts].copy()

        # roll forward k times using your one-step predictor
        for step in range(k):
            # dt for this step for each start
            if use_dt:
                dt_vec = dts[starts + step]
            else:
                # your _predict_nextstep likely ignores dt if use_dt=False, but keep the column
                dt_vec = np.zeros_like(starts, dtype=float)

            # Build a "pairs" table resembling _build_nextstep_table output
            # Columns it usually expects: sid, age_next, val_t, dt, y_true
            pairs_step = pd.DataFrame({
                "sid": sid,
                "age_next": ages[starts + step + 1],
                "val_t": cur,
                "dt": dt_vec,
                "y_true": vals[starts + step + 1],  # available for step-wise sanity; not used for chaining
            })

            # One-step predict over the full vector at once
            pred_step = _predict_nextstep(model, pairs_step, use_dt=use_dt)
            cur = pred_step["y_pred"].to_numpy(float)

        # After k steps, compare to the true value at t+k
        y_true_k    = vals[starts + k]
        age_target  = ages[starts + k]
        out.append(pd.DataFrame({
            "sid": sid,
            "age_target": age_target,
            "y_true": y_true_k,
            "y_pred": cur
        }))

    if len(out) == 0:
        return pd.DataFrame(columns=["sid", "age_target", "y_true", "y_pred"])
    return pd.concat(out, ignore_index=True)


def _merge_xy_k(px_k, py_k):
    """
    Merge x and y predictions on [sid, age_target] and suffix shared cols.
    Produces columns: y_true_x, y_pred_x, y_true_y, y_pred_y, sid, age_target
    """
    if px_k.empty or py_k.empty:
        return pd.DataFrame()
    return pd.merge(px_k, py_k, on=["sid", "age_target"], suffixes=("_x", "_y"))


# ---------------------------------------------------------------------
# Public API: per-SID and per-landmark 2D MAE for skip {1,2,4,8}
# ---------------------------------------------------------------------
def per_sid_and_per_landmark_2d_skips_from_anyfile(
    pkl_path, models_dir, data, use_dt=True, skips=(1, 2, 4, 8)
):
    """
    Returns:
      df_per_sid : rows per sid with 2D MAE for each regime (skip1/2/4/8)
        [landmark, sex, sid, regime, mae_2d, n_pairs, model_x, model_y]
      df_per_lm  : per-landmark summary (weighted by all pairs)
        [landmark, sex, regime, mae_2d, n_pairs, model_x, model_y]
    """
    # ---- parse & find twin (same as your current function) ----
    landmark, sex_key, axis = infer_meta_from_filename(Path(pkl_path).name)
    other_axis = "y" if axis == "x" else "x"

    twin_path = _find_twin_model(models_dir, landmark, sex_key, other_axis)
    if twin_path is None:
        # no 2D possible
        return (pd.DataFrame(columns=["landmark","sex","sid","regime","mae_2d","n_pairs","model_x","model_y"]),
                pd.DataFrame(columns=["landmark","sex","regime","mae_2d","n_pairs","model_x","model_y"]))

    # load both models (one for x, one for y), independent of 'axis'
    m_this, _ = load_model(pkl_path)
    m_twin, _ = load_model(twin_path)

    # name them explicitly
    if axis == "x":
        model_x, model_y = m_this, m_twin
        name_x,  name_y  = Path(pkl_path).name, Path(twin_path).name
    else:
        model_x, model_y = m_twin, m_this
        name_x,  name_y  = Path(twin_path).name, Path(pkl_path).name

    # Filter test data exactly like your current function
    df_sub = subset_for_model(data, landmark, sex_key)
    if df_sub.empty:
        return (pd.DataFrame(columns=["landmark","sex","sid","regime","mae_2d","n_pairs","model_x","model_y"]),
                pd.DataFrame(columns=["landmark","sex","regime","mae_2d","n_pairs","model_x","model_y"]))

    # ---- compute for each skip K ----
    per_sid_rows = []
    per_lm_rows  = []

    for k in skips:
        # k-step predictions for x and y using the SAME one-step models (rolled k times)
        px_k = _kstep_predict_axis(model_x, df_sub, axis_col="x", k=k, use_dt=use_dt)
        py_k = _kstep_predict_axis(model_y, df_sub, axis_col="y", k=k, use_dt=use_dt)
        merged = _merge_xy_k(px_k, py_k)
        if merged.empty:
            continue

        # 2D error for every (sid, target age)
        err2d = np.hypot(
            merged["y_pred_x"] - merged["y_true_x"],
            merged["y_pred_y"] - merged["y_true_y"]
        )
        merged = merged.assign(err2d=err2d)

        # per-SID MAE2D
        g_sid = (merged
                 .groupby("sid", as_index=False)["err2d"]
                 .agg(mae_2d="mean", n_pairs="size"))
        g_sid["landmark"] = landmark
        g_sid["sex"]      = sex_key
        g_sid["regime"]   = f"skip{k}"
        g_sid["model_x"]  = name_x
        g_sid["model_y"]  = name_y
        # rounding
        g_sid["mae_2d"]   = g_sid["mae_2d"].round(4)
        per_sid_rows.append(g_sid)

        # per-landmark (weighted over all pairs)
        #   -> average over ALL err2d (not mean-of-means)
        mae_2d_lm = merged["err2d"].mean()
        n_pairs_lm = int(merged.shape[0])
        per_lm_rows.append(pd.DataFrame({
            "landmark": [landmark],
            "sex":      [sex_key],
            "regime":   [f"skip{k}"],
            "mae_2d":   [round(float(mae_2d_lm), 4)],
            "n_pairs":  [n_pairs_lm],
            "model_x":  [name_x],
            "model_y":  [name_y],
        }))

    if len(per_sid_rows) == 0:
        df_per_sid = pd.DataFrame(columns=["landmark","sex","sid","regime","mae_2d","n_pairs","model_x","model_y"])
    else:
        df_per_sid = pd.concat(per_sid_rows, ignore_index=True)

    if len(per_lm_rows) == 0:
        df_per_lm = pd.DataFrame(columns=["landmark","sex","regime","mae_2d","n_pairs","model_x","model_y"])
    else:
        df_per_lm = pd.concat(per_lm_rows, ignore_index=True)

    return df_per_sid, df_per_lm


In [13]:
from pathlib import Path
from glob import glob
import pandas as pd

# ---- Wrapper: run all x/y model pairs over a given TEST dataframe ----
def run_skip_mae2d_over_models(models_dir, data_test, use_dt=True, skips=(1,2,4,8)):
    dfs_sid, dfs_lm = [], []
    seen = set()  # (landmark, sex) processed via the x-file only

    for p in sorted(glob(f"{models_dir}/*.pkl")):
        lm, sex_key, axis = infer_meta_from_filename(Path(p).name)
        if axis != "x":
            continue
        key = (lm, sex_key)
        if key in seen:
            continue
        seen.add(key)

        df_sid, df_lm = per_sid_and_per_landmark_2d_skips_from_anyfile(
            pkl_path=p,
            models_dir=models_dir,
            data=data_test,          # <-- pass the TEST split here
            use_dt=use_dt,
            skips=skips
        )
        if not df_sid.empty: dfs_sid.append(df_sid)
        if not df_lm.empty:  dfs_lm.append(df_lm)

    per_sid_all = pd.concat(dfs_sid, ignore_index=True) if dfs_sid else pd.DataFrame()
    per_lm_all  = pd.concat(dfs_lm,  ignore_index=True) if dfs_lm  else pd.DataFrame()
    return per_sid_all, per_lm_all

# ---- Example end-to-end with your split helpers ----
# 1) Load data and split json
# df_long must contain at least: sid, age, x, y, sex ('M'/'F'), landmark
df_long = pd.read_csv("/data/all_landmark_series_long.csv")
split_json = "/data/splits/sid_split_v1.json"
train_sids, test_sids, _ = load_sid_split(split_json)
re_splits = apply_sid_split(df_long, train_sids, test_sids)

# 2) Run for each population: all / male / female
models_dir = "./models_sep"    # change if needed
for pop in ["all", "male", "female"]:
    _, test_df = re_splits[pop]  # (train, test) tuple; we only use test
    per_sid, per_lm = run_skip_mae2d_over_models(
        models_dir=models_dir,
        data_test=test_df,
        use_dt=True,
        skips=(1,2,4,8)
    )

    # 3) Save tidy outputs
    per_sid.to_csv(f"per_sid_mae2d_{pop}_skip_1_2_4_8.csv", index=False)
    per_lm.to_csv(f"per_landmark_mae2d_{pop}_skip_1_2_4_8.csv", index=False)

    # 4) (Optional) wide views for quick tables
    if not per_lm.empty:
        wide_lm = per_lm.pivot_table(index=["landmark","sex"], columns="regime", values="mae_2d").reset_index()
        wide_lm.to_csv(f"per_landmark_mae2d_{pop}_skip_wide.csv", index=False)
    if not per_sid.empty:
        wide_sid = per_sid.pivot_table(index=["landmark","sex","sid"], columns="regime", values="mae_2d").reset_index()
        wide_sid.to_csv(f"per_sid_mae2d_{pop}_skip_wide.csv", index=False)

print("Done.")


d:\Conda\envs\dental\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\Conda\envs\dental\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator Ridge from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\Conda\envs\dental\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. U

Done.


d:\Conda\envs\dental\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\Conda\envs\dental\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator Ridge from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
d:\Conda\envs\dental\lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.7.2 when using version 1.6.1. This might lead to breaking code or invalid results. U

time interval , same for all sid